# Get stats for all tools in curated_tools.tsv, sum up the users for last 5 years. 


In [47]:
import pandas as pd
import os

COMMUNITY_PATH = "../../communities/imaging/"
CURATED_TOOLS_PATH = os.path.join(COMMUNITY_PATH, "resources", "curated_tools.tsv")
ALL_COMMUNITY_TOOLS_PATH = os.path.join(COMMUNITY_PATH, "metadata", "tool_status.tsv")
ALL_TOOLS = os.path.join("../../communities/all/", "resources", "tools.tsv")


# Load TSV file
all_tools = pd.read_csv(ALL_TOOLS, sep="\t")
c_tools = pd.read_csv(ALL_COMMUNITY_TOOLS_PATH, sep="\t")

# Filter rows where Suite ID matches
filtered_tools = all_tools[all_tools["Suite ID"].isin(c_tools["Suite ID"])]

In [ ]:
filtered_tools = filtered_tools.copy()

# Select the exact column
user_columns = ["Suite users (last 5 years) on main servers"]
#user_columns = ["Suite runs (last 5 years) on main servers"]

# Convert to numeric safely
filtered_tools[user_columns] = (
    filtered_tools[user_columns]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

# Sum total users
total_users = filtered_tools[user_columns].sum().sum()

print("Total users (last 5 years, main servers):", int(total_users))


Total users (last 5 years, main servers): 99399


# Get IT users from the raw sql query

In [51]:
import pandas as pd
import os

BASE_PATH = "../../sources/data/usage_stats/usage_stats_2025.08.31"  # adjust if needed

servers = ["eu", "fr", "org", "org.au"]

interactive_tools = [
    "interactive_tool_anylabeling",
    "interactive_tool_bellavista",
    "interactive_tool_cellpose",
    "interactive_tool_cellprofiler",
    "interactive_tool_cellxgene",
    "interactive_tool_ilastik",
    "interactive_tool_napari",
    "interactive_tool_qupath",
]

all_data = []

for server in servers:
    file_path = os.path.join(
        BASE_PATH,
        server,
        "tool_users_5y_until_2025.08.31.csv"
    )
    
    df = pd.read_csv(file_path, header=None, names=["tool_id", "users"])

    # 🔹 Ensure numeric users
    df["users"] = pd.to_numeric(df["users"], errors="coerce").fillna(0)    

    # Extract last part after "/"
    df["tool_short"] = df["tool_id"].astype(str).str.split("/").str[-1]
    
    # Keep only interactive tools
    df_filtered = df[df["tool_short"].isin(interactive_tools)].copy()
    
    df_filtered["server"] = server

    all_data.append(df_filtered)

# Combine all servers
combined = pd.concat(all_data, ignore_index=True)
print(combined)

# # Total across all interactive tools
total_users = combined["users"].sum()
print("\nTotal users across all interactive tools (5y):", total_users)


                         tool_id  users                     tool_short  server
0     interactive_tool_cellxgene  244.0     interactive_tool_cellxgene      eu
1        interactive_tool_qupath   30.0        interactive_tool_qupath      eu
2      interactive_tool_cellpose   25.0      interactive_tool_cellpose      eu
3        interactive_tool_napari   18.0        interactive_tool_napari      eu
4  interactive_tool_cellprofiler   15.0  interactive_tool_cellprofiler      eu
5       interactive_tool_ilastik   12.0       interactive_tool_ilastik      eu
6   interactive_tool_anylabeling    8.0   interactive_tool_anylabeling      eu
7    interactive_tool_bellavista    2.0    interactive_tool_bellavista      eu
8     interactive_tool_cellxgene   28.0     interactive_tool_cellxgene  org.au

Total users across all interactive tools (5y): 382.0
